In [2]:
!pip install torch_geometric
!pip install adamp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for adamp: filename=adamp-0.3.0-py3-none-any.whl size=5984 sha256=e05c5cca644c808b16c0f6f38f0d9ca944c199b784904ac8dac658b930669a74
  Stored in directory: /root/.cache/pip/wheels/c7/ad/0f/b41b1c45b18c66e5eef5d2254415af8055c7e2b0934145157d
Successfully built adamp


In [10]:
import torch
import torch.nn.functional as F
from torch_geometric.data import DataLoader
from torch_geometric.nn import GINConv, global_mean_pool
from torch_geometric.datasets import TUDataset
from torch_geometric.transforms import NormalizeFeatures
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, log_loss
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
import random
import numpy as np


# Set random seeds for reproducibility
def set_random_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Define the GIN model with pooling and regularization adjustments
class GIN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=4, dropout=0.5):
        super(GIN, self).__init__()
        self.layers = torch.nn.ModuleList()

        # Initial GIN layer
        self.layers.append(
            GINConv(
                torch.nn.Sequential(
                    torch.nn.Linear(in_channels, hidden_channels),
                    torch.nn.ReLU(),
                    torch.nn.Linear(hidden_channels, hidden_channels)
                )
            )
        )

        # Additional GIN layers
        for _ in range(num_layers - 1):
            self.layers.append(
                GINConv(
                    torch.nn.Sequential(
                        torch.nn.Linear(hidden_channels, hidden_channels),
                        torch.nn.ReLU(),
                        torch.nn.Linear(hidden_channels, hidden_channels)
                    )
                )
            )

        self.fc = torch.nn.Linear(hidden_channels, out_channels)  # Final layer for classification
        self.dropout = dropout

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv in self.layers:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        x = self.fc(F.dropout(x, p=self.dropout, training=self.training))  # Apply dropout in final layer
        return F.log_softmax(x, dim=1)

# Train function with gradient clipping
def train_model(model, loader, epochs=200, lr=0.001, weight_decay=1e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6)
    criterion = torch.nn.CrossEntropyLoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for data in loader:
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, data.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)  # Gradient clipping
            optimizer.step()
            total_loss += loss.item()

        scheduler.step(total_loss / len(loader))

    return model

# Evaluation function
def evaluate_model(model, loader, num_classes):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for data in loader:
            out = model(data)
            preds = out.argmax(dim=1)

            if data.y.ndim == 2:
                labels = data.y.argmax(dim=1)
            else:
                labels = data.y

            all_preds.append(out)
            all_labels.append(labels)

    all_preds = torch.cat(all_preds, dim=0)
    all_labels = torch.cat(all_labels, dim=0).view(-1)

    out_prob = F.softmax(all_preds, dim=1).cpu()

    current_num_classes = out_prob.shape[1]
    if current_num_classes != num_classes:
        if current_num_classes < num_classes:
            padding = num_classes - current_num_classes
            out_prob = torch.cat([out_prob, torch.zeros(out_prob.size(0), padding)], dim=1)
        else:
            out_prob = out_prob[:, :num_classes]

    auc_roc = roc_auc_score(all_labels.cpu().numpy(), out_prob[:, 1], average='weighted') if num_classes == 2 else roc_auc_score(all_labels.cpu().numpy(), out_prob, multi_class='ovr', average='weighted')
    logloss = log_loss(all_labels.cpu().numpy(), out_prob)

    test_acc = (all_preds.argmax(dim=1) == all_labels).sum().item() / len(all_labels)
    f1 = f1_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')
    precision = precision_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')
    recall = recall_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')

    return {
        'accuracy': test_acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc_roc': auc_roc,
        'log_loss': logloss
    }

# Set random seed
set_random_seed(42)

# Load and augment datasets
dataset_nci1 = TUDataset(root='/content/NCI1', name='NCI1', transform=NormalizeFeatures())
dataset_mutag = TUDataset(root='/content/MUTAG', name='MUTAG', transform=NormalizeFeatures())

# Split datasets into training and testing sets (70/30)
train_idx_nci1, test_idx_nci1 = train_test_split(range(len(dataset_nci1)), test_size=0.3, random_state=42)
train_dataset_nci1 = dataset_nci1[train_idx_nci1]
test_dataset_nci1 = dataset_nci1[test_idx_nci1]

train_idx_mutag, test_idx_mutag = train_test_split(range(len(dataset_mutag)), test_size=0.3, random_state=42)
train_dataset_mutag = dataset_mutag[train_idx_mutag]
test_dataset_mutag = dataset_mutag[test_idx_mutag]

train_loader_nci1 = DataLoader(train_dataset_nci1, batch_size=32, shuffle=True)
test_loader_nci1 = DataLoader(test_dataset_nci1, batch_size=32, shuffle=False)

train_loader_mutag = DataLoader(train_dataset_mutag, batch_size=32, shuffle=True)
test_loader_mutag = DataLoader(test_dataset_mutag, batch_size=32, shuffle=False)

# Initialize and train GIN model on NCI1
model_nci1 = GIN(in_channels=dataset_nci1.num_node_features, hidden_channels=64, out_channels=dataset_nci1.num_classes, dropout=0.5)
train_model(model_nci1, train_loader_nci1, epochs=200)

# Evaluate on NCI1 dataset
print("\nResults on NCI1 Dataset Before Transfer Learning:")
nci1_results = evaluate_model(model_nci1, test_loader_nci1, num_classes=dataset_nci1.num_classes)
print(nci1_results)

# Transfer learning to MUTAG dataset
model_mutag = GIN(in_channels=dataset_mutag.num_node_features, hidden_channels=64, out_channels=dataset_mutag.num_classes, dropout=0.5)
state_dict_nci1 = model_nci1.state_dict()
state_dict_mutag = model_mutag.state_dict()

# Transfer compatible weights
for key in state_dict_nci1:
    if key in state_dict_mutag and state_dict_nci1[key].shape == state_dict_mutag[key].shape:
        state_dict_mutag[key] = state_dict_nci1[key]

model_mutag.load_state_dict(state_dict_mutag, strict=False)
train_model(model_mutag, train_loader_mutag, epochs=300)

# Evaluate on MUTAG dataset
print("\nResults on MUTAG Dataset After Transfer Learning:")
mutag_results = evaluate_model(model_mutag, test_loader_mutag, num_classes=dataset_mutag.num_classes)
print(mutag_results)


/usr/local/lib/python3.10/dist-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)



Results on NCI1 Dataset Before Transfer Learning:


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2956: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


{'accuracy': 0.7623682076236821, 'f1': 0.7620279247065852, 'precision': 0.7626555747944295, 'recall': 0.7623682076236821, 'auc_roc': 0.825297873012863, 'log_loss': 0.6050024952174005}

Results on MUTAG Dataset After Transfer Learning:
{'accuracy': 0.8070175438596491, 'f1': 0.8083722820564926, 'precision': 0.8102493074792245, 'recall': 0.8070175438596491, 'auc_roc': 0.888888888888889, 'log_loss': 0.3877223773475985}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2956: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


In [8]:
82,82,80,80,80,80

(82, 82, 80, 80, 80)